<!-- <style> * { font-size: 100%; } </style> -->
<small> 

# rsyncd
- https://linux.die.net/man/5/rsyncd.conf
- https://superuser.com/questions/243656/how-to-configure-and-use-rsyncd


</small>

In [ ]:
## Setup Working Directory & Files

# Working Dir
WORKING_DIRECTORY="/srv/docker/rsyncd-server"
mkdir -p ${WORKING_DIRECTORY}
cd ${WORKING_DIRECTORY}

# Make Folders & Files
# mkdir -p ./{config,repositories}
# mkdir -p ./{config,secrets}
# ls -alt ${WORKING_DIRECTORY}

# Create Needed Files
# touch ${WORKING_DIRECTORY}/secrets/rsyncd.secrets

In [ ]:
# Generate User Passwords
printf 'username:%s\n' "$(openssl rand -base64 64 | head -c 64 | tr -d '\n' | tr -dc 'A-Za-z0-9')"

In [ ]:
# Generate Passwords
COUNT=12
LENGTH=64
FILTER='A-Za-z0-9' # alnum:'A-Za-z0-9'  # low:'a-z0-9'  # hex:'A-Fa-f0-9'

printf 'Password Generator | COUNT=%s | LENGTH=%s | FILTER=%s\n' \
  "$COUNT" "$LENGTH" "$FILTER"

for ((i=1; i<=COUNT; i++)); do
  openssl rand -base64 "$((LENGTH + 12))" | tr -dc 'A-Za-z0-9' | head -c "$LENGTH"
  echo
done

# Docker

## Docker Context

In [ ]:
# Use Context
# docker context use default
docker context use nas-03

## Build and Deploy Container

In [ ]:
# Build (example)
docker compose --progress plain --file docker-compose.yaml --env-file docker-compose.example.env build

In [ ]:
# Build
docker compose --progress plain --file ../docker-compose.yaml --env-file ../docker-compose.nas-03_unraid.env build

In [ ]:
# Build & Run
docker compose --progress plain --file ../docker-compose.yaml --env-file ../docker-compose.nas-03_unraid.env up --detach --build

In [49]:
# Build & Run w/ SSH
docker compose --progress plain --profile ssh --file ../docker-compose.yaml --env-file ../docker-compose.nas-03_unraid.env up --detach --build

 Image rsyncd-server-rsyncd Building 
 Image rsyncd-server-rsync-ssh Building 
#1 [internal] load local bake definitions
#1 reading from stdin 1.02kB done
#1 DONE 0.0s

#2 [rsyncd internal] load build definition from dockerfile
#2 transferring dockerfile:
#2 transferring dockerfile: 242B done
#2 DONE 0.2s

#3 [rsync-ssh internal] load build definition from dockerfile
#3 transferring dockerfile: 263B done
#3 DONE 0.2s

#4 [rsyncd internal] load metadata for docker.io/library/alpine:3.22
#4 DONE 0.4s

#5 [rsyncd internal] load .dockerignore
#5 transferring context: 2B done
#5 DONE 0.0s

#6 [rsync-ssh internal] load .dockerignore
#6 transferring context: 2B done
#6 DONE 0.0s

#7 [rsyncd 1/4] FROM docker.io/library/alpine:3.22@sha256:55ae5d250caebc548793f321534bc6a8ef1d116f334f18f4ada1b2daad3251b2
#7 DONE 0.0s

#8 [rsyncd internal] load build context
#8 transferring context: 35B done
#8 DONE 0.0s

#9 [rsync-ssh internal] load build context
#9 transferring context: 35B done
#9 DONE 0.0s

#1

In [48]:
# Build & Run in Context
CONTEXT=nas-03
# docker --context lxc-docker-01 compose --profile "*" --env-file .env.lxc-docker  up --build --detach --remove-orphans

# Build the Base Image for Minecraft Bedrock Server Backup (Bedrockifier)
docker --context "${CONTEXT}" build \
  --progress plain \
  --tag local/rsync-server:latest \
  --env-file ../docker-compose.nas-03_unraid.env up
  --file ../docker-compose.yaml \
  --profile ssh

# Build + Run the Minecraft Bedrock Server & Backup Service
docker --context "${CONTEXT}" compose \
  --ansi never \
  --env-file ../.env.lxc-docker \
  up --build --detach --remove-orphans

unknown flag: --env-file

Usage:  docker buildx build [OPTIONS] PATH | URL | -

Run 'docker buildx build --help' for more information
bash: --file: command not found
couldn't find env file: /nas-01/volume1/docker/rsync-server/.env.lxc-docker


: 1

## Rsyncd Testing

In [ ]:
# List Rsync Modules
docker exec rsyncd-server rsync rsync://127.0.0.1/

In [ ]:
# List Rsync Modules
rsync rsync://nas-03.internal

In [ ]:
# List Rsync Module Contents (example)
cat ../secrets/rsync.example.pass
chmod 600 ../secrets/rsync.example.pass
rsync --password-file=../secrets/rsync.example.pass rsync://servers_user@nas-03.internal:873/servers-main/

In [ ]:
# List Rsync Module Contents (testing)
PASS_FILE_PATH="../secrets/rsync.test.pass"
# cat "${PASS_FILE_PATH}"
chmod 600 "${PASS_FILE_PATH}"
rsync --password-file=${PASS_FILE_PATH} rsync://test_user@nas-03.internal:873/testing/

In [ ]:
# Send a test file (testing)

# Password file
PASS_FILE_PATH="../secrets/rsync.test.pass"
# cat "${PASS_FILE_PATH}"
chmod 600 "${PASS_FILE_PATH}"

# File setup
TEST_FOLDER_PATH="/tmp/rsyncd-test"
mkdir --parents "${TEST_FOLDER_PATH}"
printf 'hello\n' > "${TEST_FOLDER_PATH}/hello.txt"

# Upload
rsync --archive --verbose \
  --password-file="${PASS_FILE_PATH}" \
  "${TEST_FOLDER_PATH}/" \
  "rsync://test_user@nas-03.internal:873/testing/"